# Azure VM CPU Utilization Analysis - Data Extraction
This notebook performs stratified sampling of VM metadata and extracts corresponding CPU readings from the Azure Public Dataset.

## Step 1: Dependencies and Global Configuration
In this section, we import the necessary libraries for data manipulation, network requests, and statistical analysis. We also define global constants and file paths to ensure consistency throughout the notebook.

In [15]:
import pandas as pd
import requests
import io
import gzip
import time
import os
import glob
import json
from statsmodels.tsa.stattools import adfuller

# --- Global Configuration ---
VMTABLE_URL = "https://azurepublicdatasettraces.blob.core.windows.net/azurepublicdatasetv2/trace_data/vmtable/vmtable.csv.gz"
VMTABLE_PATH = 'vmtable.csv.gz'
READINGS_BASE_URL = "https://azurepublicdatasettraces.blob.core.windows.net/azurepublicdatasetv2/trace_data/vm_cpu_readings/vm_cpu_readings-file-{}-of-195.csv.gz"

# File Paths
CHECKPOINT_DIR = 'checkpoints'
FAILED_LOG = 'failed_files.json'
METADATA_FILE = 'deep_study_vms_metadata.csv'
FINAL_READINGS_FILE = 'final_deep_readings_20vms.csv'
REPAIR_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'readings_checkpoint_repair.csv')

# Ensure environment is ready
if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)

## Step 2: Source Metadata Acquisition
The Azure Public Dataset provides a `vmtable` containing metadata for millions of VMs. We download this compressed CSV file (approx. 226MB) if it is not already present in the local directory.

In [16]:
def download_vmtable(url, destination):
    """Downloads the VM metadata table if it does not exist."""
    if not os.path.exists(destination):
        print(f"Downloading {destination} from source...")
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(destination, 'wb') as f:
                f.write(response.content)
            print("✅ Download complete.")
        else:
            print(f"❌ Failed to download. Status code: {response.status_code}")
    else:
        print(f"✅ {destination} already exists. Skipping download.")

download_vmtable(VMTABLE_URL, VMTABLE_PATH)

✅ vmtable.csv.gz already exists. Skipping download.


## Step 3: Target Selection and Idempotent Resync
To ensure longitudinal analysis, we filter for VMs that:
1. Have a known category (Interactive or Delay-insensitive).
2. Survived for at least 7 days (604,800 seconds).
3. Maintained an average CPU utilization above 5%.

**Note:** If a processed readings file already exists, we synchronize our target list with the IDs present in that file to maintain data consistency.

In [17]:
# Selection Criteria
MIN_DURATION = 604800  # 7 Days in seconds
MIN_AVG_CPU = 5.0

# Define schema for vmtable
VMTABLE_COLS = [
    'vm_id', 'subscription_id', 'deployment_id', 
    'timestamp_vm_created', 'timestamp_vm_deleted', 
    'max_cpu', 'avg_cpu', 'p95_max_cpu', 
    'vm_category', 'vm_virtual_core_count_bucket', 'vm_memory_gb_bucket'
]

print("Loading and processing VM metadata...")
df_vm = pd.read_csv(VMTABLE_PATH, compression='gzip', names=VMTABLE_COLS)
df_vm['duration'] = df_vm['timestamp_vm_deleted'] - df_vm['timestamp_vm_created']

if os.path.exists(FINAL_READINGS_FILE):
    print(f"📊 {FINAL_READINGS_FILE} found. Resyncing with existing target IDs...")
    existing_ids = pd.read_csv(FINAL_READINGS_FILE, usecols=['vm_id'])['vm_id'].unique()
    deep_target_vms = df_vm[df_vm['vm_id'].isin(existing_ids)].copy()
else:
    print("🔍 No existing readings found. Selecting new targets based on criteria...")
    filtered_vms = df_vm[
        (df_vm['vm_category'] != 'Unknown') & 
        (df_vm['duration'] >= MIN_DURATION) & 
        (df_vm['avg_cpu'] > MIN_AVG_CPU)
    ].copy()
    
    # Pick top 20 longest-running VMs
    deep_target_vms = filtered_vms.sort_values(by='duration', ascending=False).head(20)

# Export finalized metadata
deep_target_ids = set(deep_target_vms['vm_id'].unique())
deep_target_vms.to_csv(METADATA_FILE, index=False)

print(f"✅ Target selection complete. Total: {len(deep_target_ids)} VMs.")
print(deep_target_vms[['vm_id', 'vm_category', 'avg_cpu', 'duration']].head())

Loading and processing VM metadata...
📊 final_deep_readings_20vms.csv found. Resyncing with existing target IDs...
✅ Target selection complete. Total: 20 VMs.
                                                     vm_id        vm_category  \
3        xzQ++JF1UAkh70CDhmzkiOo+DQn+E2TLErCFKEmSswv1pl...        Interactive   
1343180  AHLllP4vjsZ5okcCEA9GQ8ND+sjUlTaIcrK343h6+rIjp0...  Delay-insensitive   
1668785  539XYwpoKXyeWClkFzhqCc0IK+I+nsY1J7w4rJ87l4bcF2...  Delay-insensitive   
1668843  AO6n08ImK3pDHNzMytZ+TnBj8IQzsA3aDmNmP2i2iYrlv1...  Delay-insensitive   
1668863  de17+2nDtFaXmZiobmFTuYdjwA1N3bTf0zHk08yXFOq90J...  Delay-insensitive   

           avg_cpu  duration  
3        30.340054   2591400  
1343180   4.690616   2591400  
1668785   4.055695   2591400  
1668843   8.382702   2591400  
1668863   8.440893   2591400  


## Step 4: Batch Extraction of CPU Readings
We iterate through 195 compressed CSV files containing time-series CPU data. We filter these files to keep only the records belonging to our 20 target VMs. To prevent data loss from network interruptions, we save checkpoints every 5 files and record any failures in a JSON log.

In [18]:
CHECKPOINT_INTERVAL = 5
READING_COLS = ['timestamp', 'vm_id', 'min_cpu', 'max_cpu', 'avg_cpu']

if os.path.exists(FINAL_READINGS_FILE):
    print(f"⏭️  Bypassing Step 4: Final dataset '{FINAL_READINGS_FILE}' already exists.")
else:
    print(f"🚀 Starting extraction from 195 files...")
    temp_data_list = []
    failed_files = []
    start_time = time.time()

    for i in range(1, 196):
        url = READINGS_BASE_URL.format(i)
        try:
            response = requests.get(url, timeout=60)
            if response.status_code == 200:
                with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
                    chunks = pd.read_csv(f, names=READING_COLS, chunksize=500000)
                    for chunk in chunks:
                        filtered = chunk[chunk['vm_id'].isin(deep_target_ids)]
                        if not filtered.empty:
                            temp_data_list.append(filtered)
                
                # Checkpoint persistence
                if i % CHECKPOINT_INTERVAL == 0 and temp_data_list:
                    batch_df = pd.concat(temp_data_list, ignore_index=True)
                    cp_path = os.path.join(CHECKPOINT_DIR, f"batch_{i}.csv")
                    batch_df.to_csv(cp_path, index=False)
                    temp_data_list = [] # Flush memory buffer
                    
                    elapsed = (time.time() - start_time) / 60
                    print(f"Progress: {i}/195 | Elapsed: {elapsed:.1f} min | Saved: batch_{i}.csv")
            else:
                print(f"⚠️ HTTP {response.status_code} at file {i}")
                failed_files.append(i)
        except Exception as e:
            print(f"❌ Error at file {i}: {e}")
            failed_files.append(i)

    # Save tail fragment
    if temp_data_list:
        pd.concat(temp_data_list, ignore_index=True).to_csv(
            os.path.join(CHECKPOINT_DIR, "batch_final_fragment.csv"), index=False
        )

    # Persist failure log for Step 5
    with open(FAILED_LOG, 'w') as f:
        json.dump(failed_files, f)
    print(f"✅ Extraction finished. {len(failed_files)} files queued for repair.")

⏭️  Bypassing Step 4: Final dataset 'final_deep_readings_20vms.csv' already exists.


## Step 5: Automated Repair and Data Patching
This step parses `failed_files.json` and attempts to re-download the specific files that failed during the initial extraction. This ensures that the time-series data is as continuous as possible without re-running the entire pipeline.

In [19]:
if os.path.exists(FINAL_READINGS_FILE):
    print("⏭️  Bypassing Step 5: Final dataset already exists.")
else:
    if os.path.exists(FAILED_LOG):
        with open(FAILED_LOG, 'r') as f:
            files_to_repair = json.load(f)
        
        if files_to_repair:
            print(f"🔧 Repairing {len(files_to_repair)} files...")
            repaired_chunks = []
            successful_fixes = []

            for i in files_to_repair:
                url = READINGS_BASE_URL.format(i)
                try:
                    response = requests.get(url, timeout=60)
                    if response.status_code == 200:
                        with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
                            chunks = pd.read_csv(f, names=READING_COLS, chunksize=500000)
                            for chunk in chunks:
                                filtered = chunk[chunk['vm_id'].isin(deep_target_ids)]
                                if not filtered.empty:
                                    repaired_chunks.append(filtered)
                        print(f"✅ File {i} recovered.")
                        successful_fixes.append(i)
                except Exception as e:
                    print(f"❌ Repair failed for file {i}: {e}")
            
            if repaired_chunks:
                pd.concat(repaired_chunks, ignore_index=True).to_csv(REPAIR_CHECKPOINT, index=False)
            
            # Update log with remaining failures
            remaining = [fid for fid in files_to_repair if fid not in successful_fixes]
            with open(FAILED_LOG, 'w') as f:
                json.dump(remaining, f)
        else:
            print("📅 No files in repair queue.")

⏭️  Bypassing Step 5: Final dataset already exists.


## Step 6: Global Data Consolidation
Once all chunks and repair patches are available, we merge them into a single CSV file. Crucially, we sort the data by `vm_id` and `timestamp` to ensure the time-series sequence is preserved for future modeling.

In [21]:
if os.path.exists(FINAL_READINGS_FILE):
    print(f"✅ '{FINAL_READINGS_FILE}' already exists.")
else:
    print("Merging all checkpoints into master dataset...")
    all_files = glob.glob(os.path.join(CHECKPOINT_DIR, "*.csv"))
    
    if all_files:
        final_df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
        # Time-series sorting is vital for analysis
        final_df = final_df.sort_values(by=['vm_id', 'timestamp'])
        final_df.to_csv(FINAL_READINGS_FILE, index=False)
        print(f"🎉 Final dataset generated. Total rows: {len(final_df)}")
    else:
        print("❌ No checkpoint files found to merge.")

✅ 'final_deep_readings_20vms.csv' already exists.


## Step 7: Data Integrity and Statistical Profiling
The final step performs a comprehensive quality check. We verify the presence of missing values, duplicates, and assess the "health" of the time-series through continuity analysis and the Augmented Dickey-Fuller (ADF) test for stationarity.

In [23]:
# Reload for final verification
df_final = pd.read_csv(FINAL_READINGS_FILE)
df_meta = pd.read_csv(METADATA_FILE)

print("=== 1. Data Structure Check ===")
print(df_final.info())

print("\n=== 2. Statistics Summary ===")
print(df_final.describe())

def check_timeseries_health(vm_id, category):
    data = df_final[df_final['vm_id'] == vm_id].sort_values('timestamp')
    if len(data) < 2: return None
    
    # Gap Detection
    diffs = data['timestamp'].diff().dropna()
    gaps = diffs[diffs > 300]
    
    # Stationarity Test
    stationary = "N/A"
    try:
        res = adfuller(data['avg_cpu'])
        stationary = "Yes" if res[1] <= 0.05 else "No"
    except: pass
    
    return {
        'Category': category,
        'Points': len(data),
        'Gaps_Found': len(gaps),
        'Max_Gap_Sec': int(diffs.max()),
        'Stationary': stationary
    }

print("\n=== 3. Representative Health Audit ===")
health_stats = []
for cat in ['Interactive', 'Delay-insensitive']:
    cat_ids = df_meta[df_meta['vm_category'] == cat]['vm_id'].unique()
    # Pick the VM with the most data points
    present_ids = df_final[df_final['vm_id'].isin(cat_ids)]
    if not present_ids.empty:
        best_id = present_ids['vm_id'].value_counts().index[0]
        health_stats.append(check_timeseries_health(best_id, cat))

print(pd.DataFrame(health_stats))

# Final Duplicate Verification
dups = df_final.duplicated(subset=['vm_id', 'timestamp']).sum()
print(f"\n=== 4. Final Duplicate Check ===\nTotal duplicates: {dups}")

=== 1. Data Structure Check ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172684 entries, 0 to 172683
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   timestamp  172684 non-null  int64  
 1   vm_id      172684 non-null  object 
 2   min_cpu    172684 non-null  float64
 3   max_cpu    172684 non-null  float64
 4   avg_cpu    172684 non-null  float64
dtypes: float64(3), int64(1), object(1)
memory usage: 6.6+ MB
None

=== 2. Statistics Summary ===
          timestamp        min_cpu        max_cpu        avg_cpu
count  1.726840e+05  172684.000000  172684.000000  172684.000000
mean   1.296065e+06       4.864539      17.177082       7.705993
std    7.479173e+05       4.656196      17.650979       7.824648
min    0.000000e+00       0.167455       0.500152       0.344585
25%    6.483000e+05       2.582480       6.118988       3.813414
50%    1.296000e+06       4.027191      10.482204       5.957475
75%    1.943700e+0